# Lab 1

Load data from the provided csv file and print .head() to verify correctness of data

In [ ]:
import pandas as pd

df = pd.read_csv("data.csv")

print(df.head())
print(df.info())


    order_id  order_date    status   item_id                  sku  \
0  100360250  2020-10-29  complete  584462.0  MEFCUR59ACC78EE5459   
1  100495355  2021-04-29  canceled  803682.0  OTHPCB5AB351EF2864B   
2  100528669  2021-06-17  canceled  848906.0  MATSAM5AAB5AEE5DBA7   
3  100361718  2020-11-05    refund  586872.0  MEFMUN59ACE5A1733DA   
4  100458625  2021-03-24  canceled  753549.0  MATINF5A7D8FD8D4881   

   qty_ordered   price   value  discount_amount    total  ...          SSN  \
0          2.0   199.9   199.9             0.00   199.90  ...  680-31-0122   
1          4.0    50.0   150.0             0.00   150.00  ...  550-99-7786   
2          2.0  2877.0  2877.0           333.93  2543.07  ...  156-23-7509   
3          2.0    49.9    49.9             0.00    49.90  ...  394-33-4622   
4          2.0  1257.3  1257.3           225.73  1031.57  ...  546-99-4899   

     Phone No.     Place Name            County          City State    Zip  \
0  219-966-6285  Indianapolis         

/tmp/ipykernel_5674/284498859.py:3: DtypeWarning: Columns (0: order_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data.csv")


## Pre analisys 

Some of the columns had strange names so i slightly renamed them

In [ ]:
# Questionalble columns 
# I want to rename some columns to make them more descriptive
df.rename(columns={
    "cust_id": "customer_id",
    "bi_st": "billing_status",
    "ref_num": "reference_number",
    "sku": "product_variant_sku",
}, inplace=True)

# Change types of some columns to more appropriate ones
df = df.astype({
    "qty_ordered": "int64",
    "customer_id": "int64",
    "item_id": "int64"
})

df.to_csv("data_cleaned.csv", index=False)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 57278 entries, 0 to 57277
Data columns (total 36 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   order_id             57278 non-null  object 
 1   order_date           57278 non-null  str    
 2   status               57278 non-null  str    
 3   item_id              57278 non-null  int64  
 4   product_variant_sku  57278 non-null  str    
 5   qty_ordered          57278 non-null  int64  
 6   price                57278 non-null  float64
 7   value                57278 non-null  float64
 8   discount_amount      57278 non-null  float64
 9   total                57278 non-null  float64
 10  category             57278 non-null  str    
 11  payment_method       57278 non-null  str    
 12  billing_status       57278 non-null  str    
 13  customer_id          57278 non-null  int64  
 14  year                 57278 non-null  int64  
 15  month                57278 non-null  str    
 1

## Analysis

1) Which month had the highest average order total?


(Computes average order `total` per month to find months with the largest average order value.)

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
# per-order total in case multiple rows per order_id
if 'order_id' in df.columns:
    orders = df.groupby('order_id').agg({'total':'sum','order_date':'first'}).reset_index()
    monthly_avg = orders.groupby(orders['order_date'].dt.to_period('M'))['total'].mean()
else:
    # fallback: average line-item totals by month
    monthly_avg = df.groupby(df['order_date'].dt.to_period('M'))['total'].mean()
print('Monthly average order total (top 5):', monthly_avg.sort_values(ascending=False).head())
print('Month with highest average order total:', monthly_avg.idxmax(), monthly_avg.max())

Monthly average order total (top 5): order_date
2021-08    1833.931679
2021-07    1388.892971
2021-06    1023.189708
2021-09    1019.883284
2021-03     976.988064
Freq: M, Name: total, dtype: float64
Month with highest average order total: 2021-08 1833.931679218968


2) Which day of week has the highest average order total?

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
dow_avg = df.groupby(df['order_date'].dt.day_name())['total'].mean().sort_values(ascending=False)
print('Average total by day of week:', dow_avg)
print('Top day:', dow_avg.idxmax(), dow_avg.max())

Average total by day of week: order_date
Wednesday    905.950628
Tuesday      863.744262
Thursday     838.294187
Saturday     823.529589
Sunday       774.886462
Monday       772.377537
Friday       768.682188
Name: total, dtype: float64
Top day: Wednesday 905.9506279286452


3) Top 10 SKUs by total revenue

In [ ]:
top_skus = df.groupby('product_variant_sku')['total'].sum().sort_values(ascending=False).head(10)
print('Top 10 SKUs by revenue:', top_skus)

Top 10 SKUs by revenue: product_variant_sku
MATSAM5B10F7E2AC8C1    1.541625e+06
MATSAM59DB757FB47A2    1.046608e+06
MATSAM59DB75ADB2F80    1.034932e+06
MATSAM5B10F91A9B6AB    5.006981e+05
MATSAM5B1509B4696EA    4.762563e+05
MATSAM5A7463EE3C1A5    4.323541e+05
MATSAM5A7DBC6689BE5    3.455571e+05
MATSAM5AE2B9EB35E5D    3.111490e+05
MATSAM5A0BFFF19C40A    3.014008e+05
MATAPP59D4D1F18B52D    2.906933e+05
Name: total, dtype: float64


4) Which Region had the highest average discount percent?

In [ ]:
if 'Discount_Percent' in df.columns and 'Region' in df.columns:
    region_disc = df.groupby('Region')['Discount_Percent'].mean().sort_values(ascending=False)
    print('Average discount percent by Region:', region_disc)
    print('Top region:', region_disc.idxmax(), region_disc.max())
else:
    print('Required columns not present: Discount_Percent and/or Region')

Average discount percent by Region: Region
West         6.170451
Midwest      6.151072
Northeast    6.127320
South        5.941328
Name: Discount_Percent, dtype: float64
Top region: West 6.170451023836179


5) What percentage of orders (rows) were canceled?

In [ ]:
if 'status' in df.columns:
    pct_canceled = (df['status'] == 'canceled').mean() * 100
    print(f"Canceled rows: {pct_canceled:.2f}%")
    print(df['status'].value_counts().head())
else:
    print('Column `status` not found in df')

Canceled rows: 38.97%
status
canceled          22322
complete          17822
received          10374
order_refunded     5200
refund              771
Name: count, dtype: int64


6) Top 10 customers by total spend

In [ ]:
cust_col = 'customer_id' if 'customer_id' in df.columns else 'cust_id' if 'cust_id' in df.columns else None
if cust_col is not None:
    top_customers = df.groupby(cust_col)['total'].sum().sort_values(ascending=False).head(10)
    print('Top customers by total spend:', top_customers)
else:
    print('No customer id column found')

Top customers by total spend: customer_id
110215    399369.9
111057    332841.9
39707     246596.0
109213    238479.7
113694    234935.2
11305     198674.3
113474    177348.4
111767    173208.5
107853    171109.9
109038    167162.5
Name: total, dtype: float64


7) Average order value (AOV) by billing type (Net/Gross) or `billing_status`

In [ ]:
billing_col = 'billing_status' if 'billing_status' in df.columns else 'bi_st' if 'bi_st' in df.columns else None
if billing_col is not None:
    aov_by_billing = df.groupby(billing_col)['total'].mean().round(2).sort_values(ascending=False)
    print('AOV by billing type:', aov_by_billing)
else:
    print('No billing column found (bi_st or billing_status)')

AOV by billing type: billing_status
Gross    1189.77
Net       693.16
Valid     447.93
Name: total, dtype: float64


8) How many unique customers are there?

In [ ]:
cust_col = 'customer_id' if 'customer_id' in df.columns else 'cust_id' if 'cust_id' in df.columns else None
if cust_col is not None:
    print('Unique customers:', df[cust_col].nunique())
else:
    print('No customer id column found')

Unique customers: 27762


9) What is the most common order status?

In [ ]:
if 'status' in df.columns:
    vc = df['status'].value_counts()
    print('Top statuses:', vc.head())
    print('Most common:', vc.idxmax(), vc.max())
else:
    print('Column `status` not found')

Top statuses: status
canceled          22322
complete          17822
received          10374
order_refunded     5200
refund              771
Name: count, dtype: int64
Most common: canceled 22322


10) Average items per order (average total quantity per `order_id`)

In [ ]:
if 'order_id' in df.columns and 'qty_ordered' in df.columns:
    qty_per_order = df.groupby('order_id')['qty_ordered'].sum()
    print('Average items per order:', qty_per_order.mean().round(2))
    print('Median items per order:', qty_per_order.median())
else:
    print('Required columns `order_id` or `qty_ordered` missing')

Average items per order: 3.25
Median items per order: 2.0


11) What fraction of rows had a discount applied?

In [ ]:
if 'discount_amount' in df.columns:
    frac = (df['discount_amount'] > 0).mean() * 100
    print(f'Rows with discount: {frac:.2f}%')
else:
    print('Column `discount_amount` not present')

Rows with discount: 30.03%


12) Correlation between discount amount and order total (are bigger discounts associated with larger totals?)

In [ ]:
cols = ['discount_amount','total']
if all(c in df.columns for c in cols):
    corr = df['discount_amount'].corr(df['total'])
    print('Correlation (discount_amount vs total):', round(corr, 4))
    print('Interpretation: near 0 => weak linear relation; positive/negative show direction')
else:
    print('Required columns missing:', [c for c in cols if c not in df.columns])

Correlation (discount_amount vs total): 0.2985
Interpretation: near 0 => weak linear relation; positive/negative show direction
